In [ ]:
import sys
import pandas as pd
import numpy as np

from os import path

# make the repo root importable so `utils` resolves from notebooks/
sys.path.insert(0, path.abspath(".."))
from utils import metrics

In [11]:
DATA_DIR = '/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027'
!ls {DATA_DIR}

combine_results.csv   game_tracking_2023.csv  player_career_successes.csv
combine_tracking.csv  game_tracking_2024.csv  player_play.csv
games.csv	      game_tracking_2025.csv  players.csv


## Basic DataFrames for receiver

In [14]:
players_df = pd.read_csv(path.join(DATA_DIR, 'players.csv'))
players_df =  players_df[players_df['nfl_position'] == 'WR']
players_df = players_df[['nfl_id', 'display_name', "draft_year"]]

In [15]:
career_df = pd.read_csv(path.join(DATA_DIR, 'player_career_successes.csv'))
players_df = players_df.merge(career_df, how='left', left_on='nfl_id', right_on='nfl_id')
players_df = players_df.drop(columns=['career_defensive_snaps', 'career_special_teams_snaps'])

In [44]:
plays_df = pd.read_csv(path.join(DATA_DIR, 'player_play.csv'))
plays_df = plays_df[plays_df['nfl_id'].isin(players_df['nfl_id'])]
plays_df = plays_df[[
    "game_id", "play_id", "nfl_id", # identifiers
    "quarter", "down", "yards_to_go", "yardline_number", "game_clock", # game context
    "offense_formation", "receiver_alignment", "route_ran", "in_motion_at_ball_snap",  # offesive context
    "team_coverage_man_zone", "team_coverage_type", # defensive context
    "pass_result", "play_nullified_by_penalty", "pass_length", "yards_gained", "expected_points_added", # play result
    "target", "rec_yards", "yards_after_catch", "separation_at_pass_forward", "cushion", "expected_yards_after_catch" # receiver result
]]
plays_df = plays_df[plays_df['play_nullified_by_penalty'] == "N"].drop(columns=['play_nullified_by_penalty'])

In [20]:
len(players_df), len(plays_df)

(106, 42806)

## Minimum Samples

## Metrics

In [ ]:
metric_frames = {
    "catch_rate": metrics.catch_rate(plays_df),
    "adot": metrics.average_depth_of_target(plays_df),
    "yprr": metrics.yards_per_route_run(plays_df),
    "tprr": metrics.targets_per_route_run(plays_df),
    "epa_per_target": metrics.epa_per_target(plays_df),
    "yac_per_reception": metrics.yac_per_reception(plays_df),
    "route_distribution": metrics.route_distribution(plays_df).set_index("nfl_id"),
    "separation": metrics.average_separation(plays_df),
}

### Join metrics into players_df

In [ ]:
# Combine all metrics on nfl_id. Some frames repeat the same count columns
# (e.g. `targets`, `routes_run`), so keep only the first copy of each.
metrics_df = pd.DataFrame(index=pd.Index([], name="nfl_id"))
for name, frame in metric_frames.items():
    new_cols = frame.columns.difference(metrics_df.columns, sort=False)
    metrics_df = metrics_df.join(frame[new_cols], how="outer")

# join() raises if a metric column clashes with an existing players_df column
players_df = players_df.set_index("nfl_id").join(metrics_df, how="left").reset_index()

In [ ]:
players_df.shape